### Sampling weight adjust for random agent

In [ ]:
import numpy as np
import itertools
from collections import defaultdict
from scipy.optimize import root_scalar

# --- 1. Define State Space ---
types = ['x', 'd', 'f', 'i']
arity = 2  # max supported arity
state_space = list(map(lambda x: "".join(x), itertools.product(*([types] * arity))))
state_space += ["q"] # Absorbing state
idx_map = {x: i for i, x in enumerate(state_space)}

initial_state = "x" * arity 
termination_condition = initial_state[:-1] + "d"

# --- 2. Markov Chain Construction ---
def build_transition_matrix(w):
    # w is a single scalar controlling the terminal 'x' operator probabilities
    # w is Control variable for converging markov chain

    operator_list = { # behavior_2.cm
        # introducing variable
        "" : {
            "d" : 3 * w, # v, r, i
            "i" : 5 * w, # 1,2,3,4,5
            "f" : 5 * w, # 1.0, 2.0, 3.0, 4.0, 5.0 
        },
        # unary operator : 
        "d" : {
            "d" : 1 # flip
        },
        # binary operator : 
        "dd" : {
            "d" : 4 # add, subtract, multiply, divide
        },
        "di" : {
            "d" : 1 # ts_mean
        },
        "df" : {
            "d" : 1 # pow
        },
        "q" : { # Absorbing state
            "q" : 1
        }
    }
    
    n_states = len(state_space)
    P = np.zeros((n_states, n_states))
    
    for i in state_space:
        if i == "q":
            P[idx_map["q"], idx_map["q"]] = 1.0
            continue
            
        outputs = defaultdict(float)
        for k, v in operator_list.items():
            for kk, vv in v.items():
                if i.endswith(k):
                    base_state = i[:-len(k)] if len(k) > 0 else i
                    j = (initial_state + base_state + kk)[-arity:]
                    outputs[j] += vv
        if i == termination_condition:
            outputs["q"] += 1.0
            
        total_counts = sum(outputs.values())
        if total_counts > 0:
            for j, val in outputs.items():
                P[idx_map[i], idx_map[j]] = val / total_counts
        else:
            raise Exception("neither can be reduced nor introduced")
            P[idx_map[i], idx_map[i]] = 1.0
            
    return P

# --- 3. Compute Expected Steps ---
def calculate_expected_steps(w):
    P = build_transition_matrix(w)
    
    transient_idx = [i for i in range(len(state_space)) if P[i, i] < 1.0]
    Q = P[np.ix_(transient_idx, transient_idx)]
    
    eigenvalues = np.linalg.eigvals(Q)
    spectral_radius = np.max(np.abs(eigenvalues))
    
    # Check for divergence 
    if spectral_radius >= 0.999:
        return 1e6 
        
    I = np.eye(len(Q))
    try:
        N = np.linalg.inv(I - Q)
        t = np.dot(N, np.ones(len(Q)))
        
        start_matrix_idx = transient_idx.index(idx_map[initial_state])
        return t[start_matrix_idx]
    except (np.linalg.LinAlgError, ValueError):
        return 1e6

# --- 4. Optimization Setup ---
# Objective: difference between expected steps and target (20)
def objective_function(w):
    return calculate_expected_steps(w) - 20.0

# Execute root finding
try:
    # Bracket [0.01, 10.0] covers explosion (w->0) to immediate stopping (w->inf)
    result = root_scalar(objective_function, bracket=[0.01, 10.0], method='brentq')
    
    if result.converged:
        optimized_w = result.root
        print("\nOptimization Successful!")
        print(f"Target Expected Steps: 20.0")
        print(f"Actual Expected Steps: {calculate_expected_steps(optimized_w):.4f}")
        print(f"Optimized Control Variable w: {optimized_w:.4f}")
    else:
        print("\nRoot finding did not converge.")
        
except ValueError as e:
    print("\nOptimization failed. Target length may be outside physical limits of the defined bracket.")
    print(e)


Optimization Successful!
Target Expected Steps: 20.0
Actual Expected Steps: 20.0000
Optimized Control Variable w: 0.0866
